# Deteccion de Patrones VCP (Volatility Contraction Pattern)

## Contexto

El **VCP** (*Volatility Contraction Pattern*) es un patron de consolidacion definido por Mark Minervini
en su metodo SEPA (*Specific Entry Point Analysis*). La idea central es que, despues de un avance
significativo, una accion forma una base donde:

1. Las **contracciones de precio** (caidas desde maximo local a minimo local) son cada vez menores.
2. La **volatilidad** (medida por ATR) se comprime progresivamente.
3. El **volumen** decrece durante la formacion, indicando que la oferta se seca.
4. Cuando el precio supera el **pivote** (ultima resistencia) con volumen alto, se genera la **senal de compra**.

Este notebook implementa el pipeline completo de deteccion sobre datos reales de **NVIDIA (NVDA)**
durante ~10 anios, usando los modulos de `src/vcp_detection/heuristic/`.

### Pipeline de deteccion

```
OHLCV data
    |
    v
[1] Swing Detection (ATR ZigZag)     -> list[SwingPoint]
    |
    v
[2] Contraction Calculation           -> list[Contraction]
    |
    v
[3] Decreasing Sequence Detection     -> DecreasingSequence | None
    + Quality Filters (max_depth,
      min_total_reduction)
    |
    v
[4] ATR Compression Verification      -> ATRCompressionResult
    |
    v
[5] Volume Contraction Verification   -> VolumeContractionResult
    (volumen decreciente en la base)
    |
    v
[6] Pivot + Breakout Signal           -> VCPSignal | None
    (con confirmacion de volumen)
```

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib.ticker as mticker
import numpy as np
import pandas as pd

# Asegurar que la raiz del proyecto esta en el path
project_root = Path.cwd().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from models.configs import ATRZigZagConfig
from vcp_detection.heuristic import (
    ATRZigZagDetector,
    compute_contractions,
    contractions_to_dataframe,
    detect_breakout_signal,
    detect_decreasing_sequence,
    identify_pivot,
    run_full_vcp_pipeline,
    verify_atr_compression,
)

pd.set_option("display.float_format", "{:.4f}".format)
plt.rcParams.update({
    "figure.facecolor": "white",
    "axes.facecolor": "white",
    "axes.grid": True,
    "grid.alpha": 0.3,
    "figure.dpi": 110,
})

## 1. Carga de datos

Usamos datos diarios de NVDA desde enero 2018 hasta abril 2026 (~8 anios, ~2077 barras de trading).
El CSV contiene columnas OHLCV + VWAP + transactions.

In [ ]:
csv_path = project_root / "data" / "csv" / "PLTR.csv"
ohlc_full = pd.read_csv(csv_path, parse_dates=["date"], index_col="date")

START_DATE = None
ohlc = ohlc_full.loc[START_DATE:].copy()

print(f"Ticker:      NVDA")
print(f"Periodo:     {ohlc.index[0].date()} a {ohlc.index[-1].date()}")
print(f"Barras:      {len(ohlc)}")
print(f"Columnas:    {list(ohlc.columns)}")
ohlc.head(3)

## 2. Configuracion de parametros

Cada etapa del pipeline tiene parametros configurables. A continuacion se justifica cada eleccion.

### 2.1 Swing Detection — ATR ZigZag

| Parametro | Valor | Justificacion |
|-----------|-------|---------------|
| `atr_length` | 14 | Estandar de la industria para ATR (Average True Range). 14 barras diarias capturan ~3 semanas de volatilidad. |
| `atr_mult` | 2.0 | El precio debe revertir al menos **2x el ATR** desde un extremo para confirmar un swing. Valores menores (1.0-1.5) generan demasiados swings ruidosos; valores mayores (3.0+) pierden patrones reales. 2.0 es el balance entre sensibilidad y ruido. |
| `use_close_only` | True | Usa close para detectar extremos. |

**Por que ATR ZigZag y no Scipy Peaks?** ATR ZigZag es inherentemente **causal** — procesa barra a barra sin ver el futuro. Cada swing tiene un `confirmed_at` explicito que marca cuando se confirmo la reversion. En pruebas sobre NVDA mostro **50.2% de precision** en picos vs 27.4% de Scipy Peaks.

### 2.2 Secuencia decreciente — Tolerance + Quality Filters

| Parametro | Valor | Justificacion |
|-----------|-------|---------------|
| `method` | `"tolerance"` | Permite que una contraccion sea hasta un 10% mayor que la anterior sin romper la monotonia. En datos reales, el ruido de mercado hace que el metodo `strict` rechace VCPs validos por diferencias triviales (ej: 6.7% seguido de 6.8%). |
| `min_contractions` | 2 | Minimo para que exista un patron — Minervini describe VCPs con 2 a 5 contracciones. |
| `max_contractions` | 6 | Limite superior. Bases con mas de 6 contracciones suelen ser consolidaciones laterales, no VCPs. |
| `lookback_bars` | 126 | ~6 meses de trading. Ventana temporal para buscar contracciones. Cubre bases intermedias (8-26 semanas). |
| `tolerance` | 0.10 | 10% de margen. Si la contraccion anterior fue 8.0%, se acepta hasta 8.8% en la siguiente. |
| `max_depth_pct` | **0.25** | Contracciones individuales mayores al 25% indican una caida de Stage 4, no una base saludable. Se rechazan secuencias que contengan alguna contraccion mayor a este umbral. |
| `min_total_reduction` | **0.70** | La ultima contraccion debe ser como maximo el 70% de la primera. Esto rechaza secuencias como 17.9% -> 17.6% que son "tecnicamente decrecientes" pero no muestran compresion real de volatilidad. |
| `max_gap_between_contractions_days` | **40** | Maximo de dias calendario entre el low de una contraccion y el high de la siguiente. Rechaza patrones demasiado estirados donde las contracciones estan muy separadas en el tiempo — un VCP saludable tiene contracciones relativamente continuas. 40 dias (~2 meses) permite bases de duracion normal sin aceptar patrones fragmentados. |

### 2.3 Compresion de ATR — Ratio

| Parametro | Valor | Justificacion |
|-----------|-------|---------------|
| `method` | `"ratio"` | Compara ATR al inicio vs al final del patron: `ATR_end / ATR_start <= threshold`. Simple y directo. |
| `atr_period` | 14 | Consistente con el periodo usado en swing detection. |
| `ratio_threshold` | **0.85** | Exige una compresion de al menos 15% del ATR. El default original (0.70) rechaza ~88% de VCPs validos en equities reales; 0.95 era demasiado laxo (solo 5% de compresion). 0.85 es el balance. |

### 2.4 Contraccion de volumen durante la base

| Parametro | Valor | Justificacion |
|-----------|-------|---------------|
| `method` | `"ratio"` | Compara volumen promedio de la primera contraccion vs la ultima: `vol_last / vol_first <= threshold`. |
| `volume_column` | `"volume"` | Columna estandar de volumen. |
| `ratio_threshold` | **0.85** | El volumen promedio en la ultima contraccion debe ser como maximo 85% del volumen en la primera. Confirma que la oferta se seca durante la formacion de la base — condicion central de Minervini. |

### 2.5 Senal de Breakout — Volumen ratio

| Parametro | Valor | Justificacion |
|-----------|-------|---------------|
| `volume_method` | `"ratio"` | El volumen del dia de breakout debe ser >= 1.5x el promedio de los 50 dias previos. |
| `volume_ratio_threshold` | **1.5** | Valor clasico de Minervini: el breakout debe venir acompanado de un aumento de al menos 50% sobre el volumen medio. Esto confirma participacion institucional. |
| `volume_lookback_days` | 50 | ~10 semanas. Baseline estable que no esta contaminado por eventos puntuales. |
| `require_volume_confirmation` | True | La confirmacion de volumen es central al metodo SEPA. Solo se bypasea en mercados sin volumen centralizado (FX). |

### 2.6 Risk Management — Stop loss y trailing

El stop loss inicial se calcula con **dos metodos** y se elige el mas cercano al precio de entrada (mayor proteccion):
- **Pattern stop**: el low de la ultima contraccion del VCP (si el precio cae ahi, el patron fallo).
- **Fixed % stop**: `entry * (1 - max_stop_loss_pct)` (cap maximo de riesgo por trade).

Despues de la entrada, el stop se actualiza dinamicamente:

| Parametro | Valor | Justificacion |
|-----------|-------|---------------|
| `max_stop_loss_pct` | **0.07** | Stop porcentual maximo: 7% debajo del entry. Si el pattern stop esta mas lejos, se usa este. Minervini recomienda 5-8%. |
| `breakeven_r_multiple` | **2.0** | A 2R de ganancia, el stop sube a breakeven (entry). Despues ratchetea en escalones de 2R. |
| `trailing_sma_period` | **20** | Periodo de SMA para la condicion de distribucion. Cambiar a 50 para un trailing mas lento. |
| `trailing_volume_factor` | **1.5** | Si el volumen del dia supera 1.5x el promedio de N dias Y el close esta debajo de la SMA(N), se genera senal de salida por distribucion. |

In [ ]:
# --- Swing Detection ---
SWING_CONFIG = ATRZigZagConfig(
    atr_length=14,
    atr_mult=2.0,
    use_close_only=False,
)
swing_detector = ATRZigZagDetector(SWING_CONFIG)

# --- Decreasing Sequence ---
SEQUENCE_PARAMS = {
    "method": "tolerance",
    "min_contractions": 2,
    "max_contractions": 6,
    "lookback_bars": 126,
    "tolerance": 0.10,
    # Quality filters (mejora #3)
    "max_depth_pct": 0.35,          # rechazar contracciones individuales > 35%
    "min_total_reduction": 0.80,    # la ultima depth debe ser <= 80% de la primera
    "max_gap_between_contractions_days": None,  # max dias entre contracciones consecutivas
}

# --- ATR Compression ---
COMPRESSION_PARAMS = {
    "method": "ratio",
    "atr_period": 14,
    "ratio_threshold": 0.85,       # exige 15% de compresion minima
}

# --- Volume Contraction durante formacion (mejora #2) ---
VOLUME_CONTRACTION_PARAMS = {
    "method": "ratio",
    "volume_column": "volume",
    "ratio_threshold": 0.85,       # vol_last / vol_first <= 0.85
}

# --- Breakout Signal ---
BREAKOUT_PARAMS = {
    "volume_method": "ratio",
    "volume_ratio_threshold": 1.0,  # Minervini clasico (1.5x promedio)
    "volume_lookback_days": 50,
    "require_volume_confirmation": True,
}

# --- Risk Management ---
RISK_PARAMS = {
    "max_stop_loss_pct": 0.07,          # stop porcentual maximo (7%)
    "breakeven_r_multiple": 2.0,        # mover a breakeven a 2R de ganancia
    "trailing_sma_period": 20,          # SMA para senal de distribucion (20 o 50)
    "trailing_volume_factor": 1.5,      # factor de volumen para distribucion
}

print("Configuracion cargada.")

## 3. Ejecucion del pipeline

Ejecutamos el pipeline completo sobre todas las barras del periodo. Para cada fecha, el pipeline
evalua si existe un VCP valido con senal de compra.

> **Nota sobre senales repetidas**: el pipeline genera una senal cada dia que el close supera el
> pivote con volumen suficiente. Esto produce multiples senales consecutivas para el mismo patron
> subyacente. En la seccion 4 agrupamos estas senales en **patrones distintos** — un patron VCP
> cuenta como detectado si y solo si produce al menos una senal de compra.

In [ ]:
results = run_full_vcp_pipeline(
    ohlc=ohlc,
    swing_detector=swing_detector,
    sequence_params=SEQUENCE_PARAMS,
    compression_params=COMPRESSION_PARAMS,
    breakout_params=BREAKOUT_PARAMS,
    volume_contraction_params=VOLUME_CONTRACTION_PARAMS,  # mejora #2
)

all_signals = {dt: sig for dt, sig in results.items() if sig is not None}
print(f"Barras evaluadas:     {len(results):,}")
print(f"Senales de compra:    {len(all_signals)}")

## 4. Agrupacion en patrones distintos

Multiples senales consecutivas que comparten el mismo pivote pertenecen al mismo patron VCP.
Agrupamos por `pivot_price` y tomamos la **primera senal** de cada grupo como la entrada
operativa del patron.

In [ ]:
def group_signals_into_patterns(
    signals: dict[pd.Timestamp, "VCPSignal"],
    risk_params: dict,
    max_gap_days: int = 30,
) -> list[dict]:
    """Agrupa senales consecutivas en patrones VCP distintos.

    Dos senales pertenecen al mismo patron si estan separadas por menos de
    `max_gap_days` dias de calendario. Dentro de cada grupo, se toma la
    primera senal como entrada operativa.

    Aplica el dual stop method: max(pattern_stop, entry * (1 - max_stop_loss_pct)).
    """
    if not signals:
        return []

    max_stop_pct = risk_params.get("max_stop_loss_pct", 0.07)

    sorted_dates = sorted(signals.keys())
    patterns = []
    current_group = [sorted_dates[0]]

    for i in range(1, len(sorted_dates)):
        gap = (sorted_dates[i] - sorted_dates[i - 1]).days
        if gap <= max_gap_days:
            current_group.append(sorted_dates[i])
        else:
            patterns.append(current_group)
            current_group = [sorted_dates[i]]
    patterns.append(current_group)

    result = []
    for group in patterns:
        first_sig = signals[group[0]]
        vol_contr = first_sig.volume_contraction
        vol_ratio = (
            vol_contr.method_metrics.get("ratio_observed")
            if vol_contr is not None else None
        )

        # Dual stop: el mas cercano al entry (mayor proteccion)
        entry = first_sig.entry_price
        stop_pattern = first_sig.suggested_stop
        stop_pct = entry * (1.0 - max_stop_pct)
        effective_stop = max(stop_pattern, stop_pct)
        stop_method = "fixed_pct" if stop_pct >= stop_pattern else "pattern"
        stop_distance = (entry - effective_stop) / entry
        initial_risk = entry - effective_stop

        result.append({
            "first_signal_date": group[0],
            "last_signal_date": group[-1],
            "n_signal_days": len(group),
            "pivot_price": first_sig.pivot_price,
            "entry_price": entry,
            "stop_pattern": stop_pattern,
            "stop_pct": stop_pct,
            "effective_stop": effective_stop,
            "stop_method": stop_method,
            "stop_distance_pct": stop_distance,
            "initial_risk": initial_risk,
            "n_contractions": first_sig.metadata["n_contractions"],
            "depths_pct": first_sig.metadata["depths_pct"],
            "atr_ratio": first_sig.atr_compression.method_metrics.get("ratio_observed"),
            "vol_contr_ratio": vol_ratio,
            "signal_obj": first_sig,
        })
    return result


patterns = group_signals_into_patterns(all_signals, risk_params=RISK_PARAMS)
print(f"Patrones VCP distintos detectados: {len(patterns)}")
print(f"(de {len(all_signals)} senales totales en {len(ohlc):,} barras)\n")

summary_rows = []
for i, p in enumerate(patterns, 1):
    depths_str = " -> ".join(f"{d:.1%}" for d in p["depths_pct"])
    vol_contr_str = f"{p['vol_contr_ratio']:.3f}" if p["vol_contr_ratio"] is not None else "N/A"
    summary_rows.append({
        "#": i,
        "Senal": p["first_signal_date"].strftime("%Y-%m-%d"),
        "Entry": f"${p['entry_price']:.2f}",
        "Stop patron": f"${p['stop_pattern']:.2f}",
        "Stop %": f"${p['stop_pct']:.2f}",
        "Stop efect.": f"${p['effective_stop']:.2f}",
        "Metodo": p["stop_method"],
        "Riesgo": f"{p['stop_distance_pct']:.1%}",
        "ATR r.": f"{p['atr_ratio']:.3f}",
        "Vol c.": vol_contr_str,
    })

df_summary = pd.DataFrame(summary_rows).set_index("#")
df_summary

## 5. Interpretacion de resultados

**Lectura de la tabla anterior:**
- **Stop patron**: stop basado en el ultimo swing low del VCP (si el precio cae ahi, el patron fallo).
- **Stop %**: stop fijo a `max_stop_loss_pct` debajo del entry (cap de riesgo).
- **Stop efect.**: `max(stop patron, stop %)` — el mas cercano al entry, que es el que se usa.
- **Metodo**: cual de los dos ganó. `"pattern"` = el patron esta apretado; `"fixed_pct"` = el patron era demasiado profundo.
- **Riesgo**: distancia porcentual entre entry y stop efectivo.
- **ATR r.**: `ATR_fin / ATR_inicio`. Valores < 1.0 confirman compresion de volatilidad.
- **Vol c.**: ratio de volumen ultima/primera contraccion. Valores < 1.0 confirman oferta secandose.

## 6. Visualizacion de patrones

Para cada patron detectado, graficamos:
- **Panel superior**: precio (close) con swing points, pivote, zona de contracciones, stop efectivo, y senal de compra.
- **Panel inferior**: volumen diario con la media movil de 50 dias (baseline del filtro de breakout).

In [ ]:
def plot_vcp_pattern(
    ohlc: pd.DataFrame,
    pattern: dict,
    pattern_number: int,
    margin_bars_before: int = 40,
    margin_bars_after: int = 30,
) -> None:
    """Grafica un patron VCP con precio, contracciones, pivot, senal y volumen."""
    sig = pattern["signal_obj"]
    seq = sig.pivot_info.sequence
    contractions = seq.contractions

    # Rango del grafico: desde antes de la primera contraccion hasta despues de la senal
    pattern_start = contractions[0].high_swing.date
    signal_date = pattern["first_signal_date"]

    start_loc = max(0, ohlc.index.get_loc(pattern_start) - margin_bars_before)
    end_loc = min(len(ohlc) - 1, ohlc.index.get_loc(signal_date) + margin_bars_after)
    window = ohlc.iloc[start_loc : end_loc + 1]

    # Swing points del patron
    swing_highs_dates = [c.high_swing.date for c in contractions]
    swing_highs_prices = [c.high_swing.price for c in contractions]
    swing_lows_dates = [c.low_swing.date for c in contractions]
    swing_lows_prices = [c.low_swing.price for c in contractions]

    # Volumen media movil 50d
    vol_ma50 = ohlc["volume"].rolling(50, min_periods=1).mean()

    fig, (ax_price, ax_vol) = plt.subplots(
        2, 1, figsize=(14, 8), height_ratios=[3, 1],
        sharex=True, gridspec_kw={"hspace": 0.08},
    )

    # --- Panel de precio ---
    ax_price.plot(
        window.index, window["close"],
        color="#2c3e50", linewidth=1.2, label="Close", zorder=2,
    )

    # Contracciones como zonas sombreadas + flechas de profundidad
    colors_contraction = plt.cm.Blues(np.linspace(0.25, 0.55, len(contractions)))
    for i, c in enumerate(contractions):
        ax_price.axvspan(
            c.high_swing.date, c.low_swing.date,
            alpha=0.12, color=colors_contraction[i], zorder=0,
        )
        mid_date = c.high_swing.date + (c.low_swing.date - c.high_swing.date) / 2
        ax_price.annotate(
            f"C{i+1}\n{c.depth_pct:.1%}",
            xy=(mid_date, (c.high_swing.price + c.low_swing.price) / 2),
            fontsize=8, ha="center", va="center", color="#2c3e50",
            fontweight="bold",
        )

    # Swing highs y lows
    ax_price.scatter(
        swing_highs_dates, swing_highs_prices,
        marker="v", s=80, color="#e74c3c", zorder=4, label="Swing High",
    )
    ax_price.scatter(
        swing_lows_dates, swing_lows_prices,
        marker="^", s=80, color="#27ae60", zorder=4, label="Swing Low",
    )

    # Linea de pivote
    pivot_price = pattern["pivot_price"]
    ax_price.axhline(
        pivot_price, color="#e67e22", linestyle="--", linewidth=1.5, alpha=0.8,
        label=f"Pivot ${pivot_price:.2f}",
    )

    # Stop efectivo y stop alternativo (linea tenue)
    effective_stop = pattern["effective_stop"]
    stop_method = pattern["stop_method"]
    ax_price.axhline(
        effective_stop, color="#e74c3c", linestyle="-", linewidth=1.5, alpha=0.8,
        label=f"Stop ${effective_stop:.2f} ({stop_method})",
    )
    alt_stop = pattern["stop_pct"] if stop_method == "pattern" else pattern["stop_pattern"]
    ax_price.axhline(
        alt_stop, color="#e74c3c", linestyle=":", linewidth=0.8, alpha=0.35,
    )

    # Senal de compra
    entry_price = pattern["entry_price"]
    ax_price.scatter(
        [signal_date], [entry_price],
        marker="*", s=300, color="#f39c12", edgecolors="#e67e22",
        linewidth=1.5, zorder=5, label=f"BUY ${entry_price:.2f}",
    )

    depths_str = " -> ".join(f"{d:.1%}" for d in pattern["depths_pct"])
    ax_price.set_title(
        f"Patron VCP #{pattern_number} — NVDA — "
        f"Senal: {signal_date.strftime('%Y-%m-%d')}  |  "
        f"Contracciones: [{depths_str}]  |  "
        f"ATR ratio: {pattern['atr_ratio']:.3f}",
        fontsize=11, fontweight="bold", pad=10,
    )
    ax_price.set_ylabel("Precio (USD)", fontsize=10)
    ax_price.legend(loc="upper left", fontsize=8, framealpha=0.9)

    # --- Panel de volumen ---
    vol_window = window["volume"]
    vol_ma_window = vol_ma50.loc[window.index]

    vol_colors = [
        "#27ae60" if window["close"].iloc[i] >= window["open"].iloc[i]
        else "#e74c3c"
        for i in range(len(window))
    ]
    ax_vol.bar(
        window.index, vol_window, width=0.8, color=vol_colors, alpha=0.6, zorder=2,
    )
    ax_vol.plot(
        window.index, vol_ma_window,
        color="#3498db", linewidth=1.5, label="Vol MA(50)", zorder=3,
    )

    # Marcar volumen del dia de senal
    if signal_date in window.index:
        signal_vol = ohlc.loc[signal_date, "volume"]
        signal_ma = vol_ma50.loc[signal_date]
        vol_ratio = signal_vol / signal_ma if signal_ma > 0 else 0
        ax_vol.bar(
            [signal_date], [signal_vol], width=0.8,
            color="#f39c12", alpha=0.9, zorder=4,
            label=f"Breakout vol ({vol_ratio:.1f}x avg)",
        )

    ax_vol.set_ylabel("Volumen", fontsize=10)
    ax_vol.legend(loc="upper left", fontsize=8, framealpha=0.9)
    ax_vol.yaxis.set_major_formatter(mticker.FuncFormatter(
        lambda x, _: f"{x/1e6:.0f}M" if x >= 1e6 else f"{x/1e3:.0f}K"
    ))

    ax_vol.xaxis.set_major_formatter(mdates.DateFormatter("%b %Y"))
    ax_vol.xaxis.set_major_locator(mdates.MonthLocator(interval=1))
    fig.autofmt_xdate(rotation=30)

    plt.tight_layout()
    plt.show()

In [ ]:
for i, pattern in enumerate(patterns, 1):
    plot_vcp_pattern(ohlc, pattern, pattern_number=i)

## 7. Detalle por patron

Analisis individual de cada patron: contracciones, compresion de ATR, y contexto de la senal.

In [ ]:
for i, p in enumerate(patterns, 1):
    sig = p["signal_obj"]
    seq = sig.pivot_info.sequence

    print(f"{'='*70}")
    print(f"PATRON VCP #{i}")
    print(f"{'='*70}")
    print(f"  Fecha de senal:      {p['first_signal_date'].strftime('%Y-%m-%d')}")
    print(f"  Precio de entrada:   ${p['entry_price']:.2f}")
    print(f"  Precio de pivote:    ${p['pivot_price']:.2f}")
    print(f"  Stop efectivo:       ${p['effective_stop']:.2f} ({p['stop_distance_pct']:.1%} de riesgo)  [{p['stop_method']}]")
    print(f"    Stop por patron:   ${p['stop_pattern']:.2f}")
    print(f"    Stop por % fijo:   ${p['stop_pct']:.2f}")
    print(f"  Contracciones:       {p['n_contractions']}")
    print(f"  ATR ratio:           {p['atr_ratio']:.3f} (threshold: 0.85)")
    print()

    print("  Contracciones individuales:")
    for j, c in enumerate(seq.contractions, 1):
        print(
            f"    C{j}: {c.high_swing.date.strftime('%Y-%m-%d')} (${c.high_swing.price:.2f}) "
            f"-> {c.low_swing.date.strftime('%Y-%m-%d')} (${c.low_swing.price:.2f})  "
            f"depth={c.depth_pct:.1%}  duration={c.duration_bars} bars"
        )
    # Gaps entre contracciones
    if len(seq.contractions) >= 2:
        gaps = []
        for j in range(len(seq.contractions) - 1):
            gap = (seq.contractions[j + 1].high_swing.date - seq.contractions[j].low_swing.date).days
            gaps.append(gap)
        gaps_str = " / ".join(f"{g}d" for g in gaps)
        print(f"    Gaps entre contracciones: {gaps_str}")
    print()

    # Quality filter metrics
    qm = seq.method_metrics
    if "max_depth_observed" in qm:
        print(f"  Filtros de calidad:")
        print(f"    Max depth:         {qm['max_depth_observed']:.1%} (threshold: {qm['max_depth_threshold']:.0%})")
        if "total_reduction_observed" in qm:
            print(f"    Total reduction:   {qm['total_reduction_observed']:.2f} (threshold: {qm['total_reduction_threshold']:.2f})")
        if "max_gap_observed_days" in qm:
            print(f"    Max gap:           {qm['max_gap_observed_days']}d (threshold: {qm['max_gap_threshold_days']}d)")
        print()

    # Volume contraction during base
    vol_contr = sig.volume_contraction
    if vol_contr is not None:
        avg_vols_str = " -> ".join(f"{v:,.0f}" for v in vol_contr.avg_volumes)
        ratio_obs = vol_contr.method_metrics.get("ratio_observed", None)
        print(f"  Contraccion de volumen en la base:")
        print(f"    Metodo:            {vol_contr.method}")
        print(f"    Vol promedio/contr: {avg_vols_str}")
        if ratio_obs is not None:
            print(f"    Ratio last/first:  {ratio_obs:.3f} (threshold: {vol_contr.method_metrics.get('threshold', 'N/A')})")
        print(f"    Pasa:              {'Si' if vol_contr.passes else 'No'}")
        print()

    vol_info = sig.volume_confirmation
    if vol_info.get("applied", False):
        print(f"  Confirmacion de volumen en breakout:")
        print(f"    Metodo:            {vol_info['method']}")
        print(f"    Vol dia breakout:  {vol_info['volume_today']:,.0f}")
        print(f"    Vol promedio 50d:  {vol_info['volume_avg']:,.0f}")
        print(f"    Ratio:             {vol_info['value_observed']:.1f}x (minimo: {vol_info['threshold_value']:.1f}x)")
    print()

## 8. Panorama general: senales sobre la serie completa

Vista bird's-eye de NVDA con todos los patrones VCP marcados sobre el grafico de precio completo.

In [ ]:
fig, (ax_price, ax_vol) = plt.subplots(
    2, 1, figsize=(16, 7), height_ratios=[3, 1],
    sharex=True, gridspec_kw={"hspace": 0.08},
)

ax_price.plot(ohlc.index, ohlc["close"], color="#2c3e50", linewidth=0.8, label="NVDA Close")

for i, p in enumerate(patterns, 1):
    sig_date = p["first_signal_date"]
    entry = p["entry_price"]
    ax_price.scatter(
        [sig_date], [entry],
        marker="*", s=200, color="#f39c12", edgecolors="#e67e22",
        linewidth=1, zorder=5,
    )
    ax_price.annotate(
        f"VCP #{i}",
        xy=(sig_date, entry),
        xytext=(15, 15), textcoords="offset points",
        fontsize=9, fontweight="bold", color="#e67e22",
        arrowprops={"arrowstyle": "->", "color": "#e67e22", "lw": 1},
    )

ax_price.set_title(
    f"NVDA — Todos los patrones VCP detectados ({ohlc.index[0].strftime('%Y')}–{ohlc.index[-1].strftime('%Y')})",
    fontsize=12, fontweight="bold",
)
ax_price.set_ylabel("Precio (USD)")
ax_price.legend(loc="upper left", fontsize=9)

vol_ma50 = ohlc["volume"].rolling(50, min_periods=1).mean()
ax_vol.bar(ohlc.index, ohlc["volume"], width=1.0, color="#bdc3c7", alpha=0.5)
ax_vol.plot(ohlc.index, vol_ma50, color="#3498db", linewidth=1.0, label="Vol MA(50)")
ax_vol.set_ylabel("Volumen")
ax_vol.legend(loc="upper left", fontsize=9)
ax_vol.yaxis.set_major_formatter(mticker.FuncFormatter(
    lambda x, _: f"{x/1e6:.0f}M" if x >= 1e6 else f"{x/1e3:.0f}K"
))

ax_vol.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
ax_vol.xaxis.set_major_locator(mdates.YearLocator())

plt.tight_layout()
plt.show()

## 9. Retorno post-senal

Para evaluar la calidad de las senales, calculamos el retorno del precio en los 5, 10, 20 y 60
dias posteriores a cada senal de compra.

> **Disclaimer**: Estos retornos son indicativos y estan sujetos a **survivorship bias** (NVDA
> sobrevivio y fue un outlier positivo). No deben interpretarse como evidencia de rentabilidad
> del metodo — para eso se necesita un backtest sobre un universo amplio con gestion de riesgo.

In [ ]:
forward_windows = [5, 10, 20, 60]
return_rows = []

for i, p in enumerate(patterns, 1):
    sig_date = p["first_signal_date"]
    entry = p["entry_price"]
    sig_loc = ohlc.index.get_loc(sig_date)

    row = {"Patron": f"#{i}", "Fecha": sig_date.strftime("%Y-%m-%d"), "Entry ($)": f"{entry:.2f}"}
    for days in forward_windows:
        future_loc = sig_loc + days
        if future_loc < len(ohlc):
            future_price = ohlc["close"].iloc[future_loc]
            ret = (future_price - entry) / entry
            row[f"+{days}d"] = f"{ret:+.1%}"
        else:
            row[f"+{days}d"] = "N/A"
    return_rows.append(row)

df_returns = pd.DataFrame(return_rows).set_index("Patron")
df_returns

## 10. Simulacion de risk management

Para cada patron, simulamos el ciclo de vida del trade con las reglas de risk management configuradas:

1. **Stop inicial**: `max(pattern_stop, entry * (1 - max_stop_loss_pct))` — dual stop method.
2. **Trailing ratchet**: a cada escalon de `breakeven_r_multiple * R`, el stop sube. A 2R se mueve a breakeven, a 4R a entry+2R, etc.
3. **Exit por distribucion**: si `close < SMA(N)` y `volumen > avg_vol(N) * factor`, se sale por TREND_VIOLATION.
4. **Exit por stop**: si `close <= stop`, se sale.

La simulacion avanza dia a dia desde la senal de compra hasta que se dispara una condicion de salida o se acaban los datos.

In [ ]:
def simulate_trade(
    ohlc: pd.DataFrame,
    pattern: dict,
    risk_params: dict,
    max_hold_days: int = 252,
) -> dict:
    """Simula un trade desde la senal de compra hasta exit o fin de datos.

    Retorna dict con: exit_date, exit_price, exit_reason, duration, pnl_pct,
    r_multiple, max_r, stop_history (lista de (date, stop) para graficar).
    """
    entry_date = pattern["first_signal_date"]
    entry_price = pattern["entry_price"]
    initial_stop = pattern["effective_stop"]
    initial_risk = pattern["initial_risk"]
    step = risk_params["breakeven_r_multiple"]
    sma_period = risk_params["trailing_sma_period"]
    vol_factor = risk_params["trailing_volume_factor"]

    entry_loc = ohlc.index.get_loc(entry_date)
    end_loc = min(entry_loc + max_hold_days, len(ohlc) - 1)

    stop = initial_stop
    stop_history = [(entry_date, stop)]
    max_r = 0.0
    exit_reason = "open"

    for loc in range(entry_loc + 1, end_loc + 1):
        dt = ohlc.index[loc]
        close = float(ohlc["close"].iloc[loc])
        profit = close - entry_price

        # Update R-multiple tracking
        if initial_risk > 0:
            r_mult = profit / initial_risk
            max_r = max(max_r, r_mult)

            # Trailing ratchet
            steps_completed = int(r_mult / step)
            if steps_completed >= 1:
                new_stop = entry_price + (steps_completed - 1) * step * initial_risk
                if new_stop > stop:
                    stop = new_stop

        stop_history.append((dt, stop))

        # Check stop hit
        if close <= stop:
            if stop >= entry_price - 1e-10:
                exit_reason = "trailing_stop"
            else:
                exit_reason = "stop_loss"
            return {
                "exit_date": dt,
                "exit_price": close,
                "exit_reason": exit_reason,
                "duration_days": (dt - entry_date).days,
                "pnl_pct": (close - entry_price) / entry_price,
                "r_multiple": profit / initial_risk if initial_risk > 0 else 0,
                "max_r": max_r,
                "stop_history": stop_history,
            }

        # Check distribution signal (SMA + volume)
        if loc >= sma_period:
            sma_slice = ohlc["close"].iloc[loc - sma_period + 1 : loc + 1]
            sma_val = float(sma_slice.mean())
            if close < sma_val and "volume" in ohlc.columns:
                vol_slice = ohlc["volume"].iloc[loc - sma_period + 1 : loc + 1]
                avg_vol = float(vol_slice.mean())
                cur_vol = float(ohlc["volume"].iloc[loc])
                if avg_vol > 0 and cur_vol > avg_vol * vol_factor:
                    return {
                        "exit_date": dt,
                        "exit_price": close,
                        "exit_reason": "distribution",
                        "duration_days": (dt - entry_date).days,
                        "pnl_pct": (close - entry_price) / entry_price,
                        "r_multiple": profit / initial_risk if initial_risk > 0 else 0,
                        "max_r": max_r,
                        "stop_history": stop_history,
                    }

    # No exit — posicion abierta al final de los datos
    last_close = float(ohlc["close"].iloc[end_loc])
    last_profit = last_close - entry_price
    return {
        "exit_date": ohlc.index[end_loc],
        "exit_price": last_close,
        "exit_reason": "open",
        "duration_days": (ohlc.index[end_loc] - entry_date).days,
        "pnl_pct": (last_close - entry_price) / entry_price,
        "r_multiple": last_profit / initial_risk if initial_risk > 0 else 0,
        "max_r": max_r,
        "stop_history": stop_history,
    }


# Correr simulacion para cada patron
trade_results = []
for i, p in enumerate(patterns, 1):
    result = simulate_trade(ohlc, p, RISK_PARAMS)
    result["pattern_num"] = i
    result["entry_date"] = p["first_signal_date"]
    result["entry_price"] = p["entry_price"]
    result["stop_method"] = p["stop_method"]
    result["initial_risk_pct"] = p["stop_distance_pct"]
    trade_results.append(result)

# Tabla resumen
sim_rows = []
for r in trade_results:
    sim_rows.append({
        "#": r["pattern_num"],
        "Entry": r["entry_date"].strftime("%Y-%m-%d"),
        "Exit": r["exit_date"].strftime("%Y-%m-%d"),
        "Razon": r["exit_reason"],
        "Dias": r["duration_days"],
        "Entry $": f"{r['entry_price']:.2f}",
        "Exit $": f"{r['exit_price']:.2f}",
        "P&L": f"{r['pnl_pct']:+.1%}",
        "R-mult": f"{r['r_multiple']:+.1f}R",
        "Max R": f"{r['max_r']:.1f}R",
        "Stop ini": r["stop_method"],
    })

df_sim = pd.DataFrame(sim_rows).set_index("#")
print(f"Simulacion de {len(trade_results)} trades con RISK_PARAMS:")
print(f"  max_stop_loss_pct = {RISK_PARAMS['max_stop_loss_pct']:.0%}")
print(f"  breakeven_r_multiple = {RISK_PARAMS['breakeven_r_multiple']}")
print(f"  trailing_sma_period = {RISK_PARAMS['trailing_sma_period']}")
print(f"  trailing_volume_factor = {RISK_PARAMS['trailing_volume_factor']}")
print()
df_sim

In [ ]:
def plot_trade_simulation(
    ohlc: pd.DataFrame,
    pattern: dict,
    trade_result: dict,
    pattern_number: int,
    risk_params: dict,
    margin_bars_before: int = 10,
) -> None:
    """Grafica la evolucion del stop durante el trade."""
    entry_date = pattern["first_signal_date"]
    exit_date = trade_result["exit_date"]
    entry_price = pattern["entry_price"]
    initial_risk = pattern["initial_risk"]

    # Ventana: desde antes del entry hasta el exit
    entry_loc = ohlc.index.get_loc(entry_date)
    exit_loc = ohlc.index.get_loc(exit_date)
    start_loc = max(0, entry_loc - margin_bars_before)
    window = ohlc.iloc[start_loc : exit_loc + 5]

    # Stop history como series
    stop_dates, stop_prices = zip(*trade_result["stop_history"])

    fig, (ax_price, ax_vol) = plt.subplots(
        2, 1, figsize=(14, 7), height_ratios=[3, 1],
        sharex=True, gridspec_kw={"hspace": 0.08},
    )

    # Precio
    ax_price.plot(
        window.index, window["close"],
        color="#2c3e50", linewidth=1.2, label="Close",
    )

    # Stop evolution
    ax_price.step(
        stop_dates, stop_prices, where="post",
        color="#e74c3c", linewidth=2.0, alpha=0.8, label="Stop loss",
    )

    # Entry
    ax_price.axhline(
        entry_price, color="#95a5a6", linestyle=":", linewidth=0.8, alpha=0.5,
    )
    ax_price.scatter(
        [entry_date], [entry_price],
        marker="*", s=250, color="#f39c12", edgecolors="#e67e22",
        linewidth=1.5, zorder=5, label=f"BUY ${entry_price:.2f}",
    )

    # Exit
    exit_colors = {
        "stop_loss": "#e74c3c",
        "trailing_stop": "#e67e22",
        "distribution": "#9b59b6",
        "open": "#3498db",
    }
    exit_markers = {
        "stop_loss": "X",
        "trailing_stop": "X",
        "distribution": "D",
        "open": "o",
    }
    reason = trade_result["exit_reason"]
    ax_price.scatter(
        [exit_date], [trade_result["exit_price"]],
        marker=exit_markers.get(reason, "o"), s=200,
        color=exit_colors.get(reason, "#7f8c8d"),
        edgecolors="black", linewidth=1, zorder=5,
        label=f"EXIT: {reason} ${trade_result['exit_price']:.2f}",
    )

    # R-level annotations
    step = risk_params["breakeven_r_multiple"]
    for r_level in range(1, 8):
        r_price = entry_price + r_level * step * initial_risk
        if r_price < window["close"].max() * 1.1:
            ax_price.axhline(
                r_price, color="#27ae60", linestyle=":", linewidth=0.5, alpha=0.3,
            )
            ax_price.text(
                window.index[-1], r_price, f" {r_level * step:.0f}R",
                fontsize=7, color="#27ae60", va="center",
            )

    pnl_str = f"{trade_result['pnl_pct']:+.1%}"
    r_str = f"{trade_result['r_multiple']:+.1f}R"
    ax_price.set_title(
        f"Trade #{pattern_number} — {reason.upper()} — "
        f"P&L: {pnl_str} ({r_str}) — "
        f"{trade_result['duration_days']}d — "
        f"Max: {trade_result['max_r']:.1f}R",
        fontsize=11, fontweight="bold", pad=10,
    )
    ax_price.set_ylabel("Precio (USD)")
    ax_price.legend(loc="upper left", fontsize=8, framealpha=0.9)

    # Volumen + SMA
    sma_period = risk_params["trailing_sma_period"]
    vol_ma = ohlc["volume"].rolling(sma_period, min_periods=1).mean()

    vol_colors = [
        "#27ae60" if window["close"].iloc[i] >= window["open"].iloc[i]
        else "#e74c3c"
        for i in range(len(window))
    ]
    ax_vol.bar(window.index, window["volume"], width=0.8, color=vol_colors, alpha=0.5)
    ax_vol.plot(
        window.index, vol_ma.loc[window.index],
        color="#3498db", linewidth=1.2, label=f"Vol MA({sma_period})",
    )

    # Threshold de distribucion
    vol_threshold = vol_ma.loc[window.index] * risk_params["trailing_volume_factor"]
    ax_vol.plot(
        window.index, vol_threshold,
        color="#9b59b6", linewidth=0.8, linestyle="--", alpha=0.5,
        label=f"{risk_params['trailing_volume_factor']}x avg (distrib.)",
    )

    ax_vol.set_ylabel("Volumen")
    ax_vol.legend(loc="upper left", fontsize=8, framealpha=0.9)
    ax_vol.yaxis.set_major_formatter(mticker.FuncFormatter(
        lambda x, _: f"{x/1e6:.0f}M" if x >= 1e6 else f"{x/1e3:.0f}K"
    ))
    ax_vol.xaxis.set_major_formatter(mdates.DateFormatter("%b %Y"))
    fig.autofmt_xdate(rotation=30)

    plt.tight_layout()
    plt.show()


for i, (p, r) in enumerate(zip(patterns, trade_results), 1):
    plot_trade_simulation(ohlc, p, r, pattern_number=i, risk_params=RISK_PARAMS)

## 11. Resumen

### Resultados
- **Periodo analizado**: ~10 anios de NVDA (2016-2026), ~2,580 barras de trading.
- **Patrones VCP con senal de compra**: se reportan en la tabla de la seccion 4.
- **Definicion de patron**: un VCP cuenta si y solo si genera al menos una senal de compra
  (close > pivot + volumen >= 1.5x promedio 50d).

### Configuracion utilizada
| Componente | Metodo | Parametros clave |
|------------|--------|------------------|
| Swing detection | ATR ZigZag | `atr_mult=2.0`, `atr_length=14` |
| Monotonia | Tolerance | `tolerance=0.10`, `min_contractions=2`, `lookback=126` |
| Quality filters | max_depth + total_reduction | `max_depth_pct=0.25`, `min_total_reduction=0.70` |
| Compresion ATR | Ratio | `threshold=0.85` |
| Vol. contraccion base | Ratio | `threshold=0.85` |
| Breakout | Volumen ratio | `ratio=1.5x`, `lookback=50d` |
| **Stop inicial** | **Dual method** | `max(pattern_low, entry * (1 - 7%))` |
| **Trailing** | **Ratchet por R** | `breakeven a 2R, escalones de 2R` |
| **Exit distribucion** | **SMA + volumen** | `close < SMA(20) y vol > 1.5x avg` |

### Filtros aplicados (6 capas)
1. **Contracciones decrecientes** con tolerancia del 10%.
2. **Max depth**: ninguna contraccion individual > 25% (rechaza breakdowns de Stage 4).
3. **Min total reduction**: la ultima contraccion debe ser <= 70% de la primera (rechaza patrones sin compresion real).
4. **ATR compression**: la volatilidad al final del patron debe ser <= 85% de la inicial.
5. **Volume contraction**: el volumen en la ultima contraccion debe ser <= 85% del de la primera (oferta secandose).
6. **Breakout volume**: el volumen del dia de senal debe ser >= 1.5x el promedio de 50 dias (demanda institucional).

### Risk management (3 capas)
1. **Stop inicial dual**: `max(last_low, entry * 0.93)` — el mas cercano al entry protege mas.
2. **Trailing ratchet**: a 2R breakeven, a 4R se bloquea 2R de ganancia, etc. Nunca baja.
3. **Exit por distribucion**: precio bajo SMA + volumen alto = senal de venta institucional.

### Limitaciones
1. **Survivorship bias**: NVDA es una accion que sobrevivio y tuvo retornos excepcionales. Los retornos post-senal estan inflados.
2. **Sin filtro de Stage 2**: este detector no verifica que NVDA este en tendencia alcista (Stage 2 de Weinstein). Eso es responsabilidad del modulo de stage detection (`src/stage_detection/`), aun no integrado.
3. **Single ticker**: para validar el metodo se necesita un backtest sobre un universo amplio de acciones con gestion de riesgo.
4. **Simulacion simplificada**: la simulacion de trades no incluye slippage, comisiones, ni capacidad limitada de posiciones.